# Phân tích Silver → Gold trên MinIO

Chọn kernel Docker Jupyter (Spark **3.5.x**) tại `http://localhost:8888`, **Restart Kernel** một lần sau khi cập nhật notebook, rồi chạy từ trên xuống.
Cell khởi tạo bổ sung Delta Lake 3.3.3 và S3A; lần đầu cần mạng để tải thư viện. CPU và bộ nhớ Spark dùng cấu hình của môi trường.

Notebook đọc **snapshot Delta** của `silver/fact_listen`, giữ sáu nhóm phân tích bên dưới và xuất chín bảng Parquet sang Gold. Không cần đọc lại Bronze hay `listen_versions`.

Cấu hình đọc từ `.env` của dự án; biến môi trường được ưu tiên:
- `MINIO_ENDPOINT`, `MINIO_SECURE`, `MINIO_ACCESS_KEY`, `MINIO_SECRET_KEY` (hoặc `MINIO_ROOT_USER`, `MINIO_ROOT_PASSWORD`). Endpoint phải truy cập được từ kernel; `localhost` trong container là chính container đó.
- `ANALYSIS_MINIO_ENDPOINT`: tùy chọn ghi đè endpoint chỉ cho notebook, ví dụ `http://minio:9000` khi cùng mạng Docker.
- `MINIO_SILVER_BUCKET=silver`, `MINIO_SILVER_PREFIX=`: nơi pipeline đã đồng bộ các bảng Delta.
- `MINIO_GOLD_BUCKET=gold`, `MINIO_GOLD_PREFIX=batch_analysis`: nơi lưu kết quả. Tài khoản cần quyền đọc Silver, ghi Gold; notebook tạo bucket Gold nếu chưa có và tài khoản có quyền tạo.

Mỗi lần xuất tạo thư mục riêng. Chỉ dùng lần xuất có `_manifest.json` với `status=succeeded`; file này được ghi sau khi đọc lại kiểm tra đủ các bảng. Thời gian phân tích và mã lần xuất đều theo UTC.

Tham khảo cấu hình: [Delta Lake / Spark](https://docs.delta.io/releases/), [Delta quick start](https://docs.delta.io/quick-start/).

In [ ]:
# Cài thư viện nhỏ vào đúng kernel khi còn thiếu; Spark có sẵn trong image.
import importlib.util

missing = [package for module, package in
           [("dotenv", "python-dotenv"), ("minio", "minio>=7.2,<8")]
           if importlib.util.find_spec(module) is None]
if missing:
    get_ipython().run_line_magic("pip", "install -q " + " ".join(missing))

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlsplit
from dotenv import dotenv_values
from minio import Minio
import pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession, functions as F, Window

env_candidates = [Path("/home/jovyan/work/.env"),
                  *(parent / ".env" for parent in [Path.cwd(), *Path.cwd().parents])]
env_path = next((path for path in env_candidates if path.is_file()), None)
settings = {**(dotenv_values(env_path) if env_path else {}), **os.environ}
access_key = settings.get("MINIO_ACCESS_KEY") or settings.get("MINIO_ROOT_USER")
secret_key = settings.get("MINIO_SECRET_KEY") or settings.get("MINIO_ROOT_PASSWORD")
if not access_key or not access_key.strip() or not secret_key or not secret_key.strip():
    raise ValueError("Cần MINIO_ACCESS_KEY/MINIO_SECRET_KEY hoặc MINIO_ROOT_USER/MINIO_ROOT_PASSWORD trong .env.")

endpoint_raw = settings.get("ANALYSIS_MINIO_ENDPOINT") or settings.get("MINIO_ENDPOINT", "localhost:9000")
endpoint_raw = endpoint_raw.strip().rstrip("/")
secure = str(settings.get("MINIO_SECURE", "false")).lower() in ("true", "1", "yes")
endpoint_url = endpoint_raw if "://" in endpoint_raw else ("https://" if secure else "http://") + endpoint_raw
parsed_endpoint = urlsplit(endpoint_url)
if (parsed_endpoint.scheme not in ("http", "https") or not parsed_endpoint.hostname
        or parsed_endpoint.path or parsed_endpoint.query or parsed_endpoint.fragment
        or parsed_endpoint.username or parsed_endpoint.password):
    raise ValueError("Endpoint MinIO phải có dạng host:port hoặc http(s)://host:port, không chứa đường dẫn hay mật khẩu.")
secure = parsed_endpoint.scheme == "https"
minio_client = Minio(parsed_endpoint.netloc, access_key=access_key, secret_key=secret_key, secure=secure)

silver_bucket = settings.get("MINIO_SILVER_BUCKET") or "silver"
silver_prefix = (settings.get("MINIO_SILVER_PREFIX") or "").strip("/")
gold_bucket = settings.get("MINIO_GOLD_BUCKET") or "gold"
gold_prefix = (settings.get("MINIO_GOLD_PREFIX") or "batch_analysis").strip("/")
SILVER_PATH = "/".join(part for part in [f"s3a://{silver_bucket}", silver_prefix, "fact_listen"] if part)

if tuple(pyspark.__version__.split(".")[:2]) != ("3", "5"):
    raise RuntimeError("Notebook này dùng Spark 3.5.x + Delta 3.3.3. Chọn kernel spark-notebook của dự án.")
packages = "io.delta:delta-spark_2.12:3.3.3,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262"
active_context = SparkContext._active_spark_context
if active_context is not None and "delta-spark" not in active_context.getConf().get("spark.jars.packages", ""):
    raise RuntimeError("Spark đang chạy chưa có Delta. Hãy Restart Kernel rồi chạy lại từ cell đầu.")

s3a_options = {
    "fs.s3a.endpoint": endpoint_url,
    "fs.s3a.access.key": access_key,
    "fs.s3a.secret.key": secret_key,
    "fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    "fs.s3a.path.style.access": "true",
    "fs.s3a.connection.ssl.enabled": str(secure).lower(),
    "fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
}
builder = (
    SparkSession.builder.appName("silver-to-gold-analysis")
    .config("spark.shuffle.sort.bypassMergeThreshold", "0")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.jars.packages", packages)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
for key, value in s3a_options.items():
    builder = builder.config("spark.hadoop." + key, value)
spark = builder.getOrCreate()
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for key, value in s3a_options.items():
    hadoop_conf.set(key, value)
hadoop_conf.setBoolean("fs.s3a.impl.disable.cache", True)

print("Spark:", spark.version, "| Silver:", SILVER_PATH)
print("Thông tin xác thực được đọc từ môi trường/.env và không hiển thị.")

In [ ]:
# Chốt một phiên bản Delta để các bảng Gold cùng đọc một snapshot Silver.
# Chạy lại từ cell này để lấy phiên bản Silver mới nhất.
if "plays" in globals():
    plays.unpersist()
quoted_silver_path = SILVER_PATH.replace("`", "``")
silver_version = int(spark.sql(f"DESCRIBE HISTORY delta.`{quoted_silver_path}` LIMIT 1").first()["version"])
silver = spark.read.format("delta").option("versionAsOf", silver_version).load(SILVER_PATH)
silver.printSchema()
print("Phiên bản Silver:", silver_version)

In [ ]:
required_columns = {"event_id", "user_id", "listened_at", "track_id", "track_name",
                    "artist_id", "artist_name", "album_id", "tags", "tag_source"}
missing_columns = required_columns - set(silver.columns)
if missing_columns:
    raise ValueError(f"fact_listen thiếu cột: {sorted(missing_columns)}. Kiểm tra đúng bảng Silver.")
print("Số file dữ liệu trong snapshot:", len(silver.inputFiles()))

## 1. Từ Silver `fact_listen` đến bảng lượt nghe `plays`

Mỗi dòng Silver là một lượt nghe đã được pipeline kiểm tra và khử trùng. Dùng nguyên khóa `user_id` (đặt tên `user` để giữ các phân tích hiện có), `event_id`, `track_id`, `artist_id`, `album_id`. `listened_at` là Unix giây, ánh xạ thành `uts`.

Ngày, giờ, thứ, tháng và tuần được tính lại theo UTC. Notebook dừng nếu Silver rỗng, sai khóa/thời gian hoặc trùng `event_id`, để sửa dữ liệu ở pipeline Silver thay vì âm thầm đổi số liệu. Không dùng các bộ đếm `playcount` tích lũy để đếm lượt nghe.

In [ ]:
import csv
import sys
from pyspark import StorageLevel


def show_all(frame):
    """In tiêu đề và từng dòng của bảng theo thứ tự hiện tại."""
    writer = csv.writer(sys.stdout, delimiter="\t", lineterminator="\n")
    writer.writerow(frame.columns)
    formatted = frame.select(*(F.col(c).cast("string") for c in frame.columns))
    for row in formatted.toLocalIterator():
        writer.writerow("NULL" if value is None else value for value in row)


spark.conf.set("spark.sql.session.timeZone", "UTC")
silver_checks = silver.agg(
    F.count("*").alias("rows"), F.countDistinct("event_id").alias("event_ids"),
    F.sum(F.when(
        F.col("event_id").isNull() | (F.length(F.trim("event_id")) == 0)
        | F.col("user_id").isNull() | (F.length(F.trim("user_id")) == 0)
        | F.col("listened_at").isNull() | ~F.col("listened_at").between(0, 253402300799),
        1).otherwise(0)).alias("invalid_rows"),
).first()
n_plays = silver_checks["rows"]
if n_plays == 0:
    raise ValueError("Silver fact_listen rỗng. Chạy và đồng bộ pipeline Silver trước khi phân tích.")
if silver_checks["invalid_rows"] or silver_checks["event_ids"] != n_plays:
    raise ValueError("Silver có khóa/thời gian không hợp lệ hoặc event_id trùng. Kiểm tra Quality và sửa pipeline Silver.")

if "plays" in globals():
    plays.unpersist()
plays = (
    silver.select("event_id", F.col("user_id").alias("user"), F.col("listened_at").alias("uts"),
                  "track_id", F.col("track_name").alias("track"),
                  "artist_id", F.col("artist_name").alias("artist"), "album_id", "tags", "tag_source")
    .withColumn("ts_utc", F.timestamp_seconds("uts"))
    .withColumn("ngay_utc", F.to_date("ts_utc"))
    .withColumn("gio_utc", F.hour("ts_utc"))
    .withColumn("thu_iso", F.weekday("ts_utc") + 1)
    .withColumn("thang", F.month("ts_utc"))
    .withColumn("cuoi_tuan", (F.col("thu_iso") >= 6).cast("int"))
    .withColumn("tuan_utc", F.to_date(F.date_trunc("week", "ts_utc")))
    .persist(StorageLevel.DISK_ONLY)
)
print(f"{n_plays} lượt nghe từ Silver phiên bản {silver_version}")
show_all(plays.orderBy("uts", "event_id"))

## 2. Mỗi user xuất hiện trong bao nhiêu ngày?

Đọc `so_ngay_co_nghe` cùng ngày đầu, ngày cuối và số ngày lịch giữa hai mốc (tính cả hai đầu). Ví dụ, 2 ngày có nghe trong khoảng 30 ngày là tín hiệu cần đối chiếu với nhật ký crawl trước khi kết luận về phiên hoặc xu hướng. Một user chỉ xuất hiện một ngày có thể đạt 100% trên khoảng quan sát rất ngắn. Scrobble riêng lẻ không cho biết một ngày trống là do user nghỉ nghe hay do thiếu dữ liệu.

In [ ]:
coverage = (
    plays.groupBy("user")
    .agg(F.count("*").alias("so_luot"),
         F.countDistinct("ngay_utc").alias("so_ngay_co_nghe"),
         F.min("ngay_utc").alias("ngay_dau"), F.max("ngay_utc").alias("ngay_cuoi"))
    .withColumn("so_ngay_lich", F.datediff("ngay_cuoi", "ngay_dau") + 1)
    .withColumn("pct_ngay_co_nghe", F.round(100 * F.col("so_ngay_co_nghe") / F.col("so_ngay_lich"), 1))
)
show_all(coverage.orderBy("pct_ngay_co_nghe", "so_ngay_co_nghe", "user"))
n_users = coverage.count()
if n_users == 0:
    raise ValueError("Silver không có lượt nghe để phân tích. Kiểm tra fact_listen.")

## 3. Giờ UTC, thứ, tháng và cuối tuần

Tính tỷ lệ của từng user trước. Ví dụ, với 2 user, một người có 1/2 lượt lúc 20h và người kia có 9/10 lượt lúc 20h, kết quả ở 20h là (50% + 90%)/2 = **70%**. Mẫu số để lấy trung bình là **tất cả user** trong `plays`. User thiếu một khung giờ đóng góp 0 vào khung đó.

Mỗi bảng có đủ các nhóm lịch, tổng `share_pct` xấp xỉ 100% sau làm tròn. `so_luot` và `so_user_co_nghe` giúp kiểm tra kích thước mẫu. `thu_iso` chạy từ 1 (thứ Hai) đến 7 (Chủ nhật). `thang` là tháng trong năm, gộp các năm nếu có. Chỉ diễn giải mùa vụ khi dữ liệu phủ đủ các tháng.

Bảng cuối so sánh tỷ trọng lượt nghe của thứ Hai–thứ Sáu với thứ Bảy–Chủ nhật. Hai nhóm gồm 5 ngày và 2 ngày mỗi tuần, vì vậy tỷ trọng này chưa cho biết trung bình **mỗi ngày** cuối tuần nghe nhiều hay ít hơn.

In [ ]:
def distribution(dimension, start, end):
    user_bins = (
        plays.groupBy("user", dimension).count()
        .join(coverage.select("user", "so_luot"), "user")
        .withColumn("user_share", F.col("count") / F.col("so_luot"))
    )
    totals = user_bins.groupBy(dimension).agg(
        F.sum("count").alias("so_luot"), F.count("*").alias("so_user_co_nghe"),
        F.round(100 * F.sum("user_share") / F.lit(n_users), 2).alias("share_pct"),
    )
    bins = spark.range(start, end + 1).select(F.col("id").alias(dimension))
    return bins.join(totals, dimension, "left").fillna(0).orderBy(dimension)

by_hour = distribution("gio_utc", 0, 23)
by_weekday = distribution("thu_iso", 1, 7)
by_month = distribution("thang", 1, 12)
by_day_type = distribution("cuoi_tuan", 0, 1).withColumn(
    "loai_ngay", F.when(F.col("cuoi_tuan") == 1, "Thứ Bảy–Chủ nhật").otherwise("Thứ Hai–thứ Sáu")
)

for table in (by_hour, by_weekday, by_month, by_day_type):
    show_all(table)

**Một ngày cuối tuần có nhiều lượt hơn một ngày thường hay không?** Với mỗi user có dữ liệu ở cả hai loại ngày, tính số lượt trung bình trên một **ngày có nghe** của từng loại. Cân hai mức này thành tỷ lệ sao cho tổng bằng 100%, rồi lấy trung bình đều giữa các user đủ điều kiện. Nếu `pct_cuoi_tuan` lớn hơn 50%, nhóm này có xu hướng nghe nhiều hơn trên một ngày cuối tuần có hoạt động. Ngày không có bản ghi vẫn chưa biết độ phủ crawl nên không đưa vào mẫu số. Nếu `so_user_so_sanh` bằng 0, chưa thể so sánh trên mẫu hiện tại.

In [ ]:
per_active_day = (
    plays.groupBy("user", "cuoi_tuan")
    .agg((F.count("*") / F.countDistinct("ngay_utc")).alias("luot_moi_ngay_co_nghe"))
    .groupBy("user")
    .agg(F.max(F.when(F.col("cuoi_tuan") == 0, F.col("luot_moi_ngay_co_nghe"))).alias("ngay_thuong"),
         F.max(F.when(F.col("cuoi_tuan") == 1, F.col("luot_moi_ngay_co_nghe"))).alias("cuoi_tuan"))
    .dropna(subset=["ngay_thuong", "cuoi_tuan"])
    .withColumn("share_cuoi_tuan", F.col("cuoi_tuan") / (F.col("cuoi_tuan") + F.col("ngay_thuong")))
)
day_comparison = per_active_day.agg(
    F.count("*").alias("so_user_so_sanh"),
    F.round(100 * F.avg(1 - F.col("share_cuoi_tuan")), 2).alias("pct_ngay_thuong"),
    F.round(100 * F.avg("share_cuoi_tuan"), 2).alias("pct_cuoi_tuan"),
)
show_all(day_comparison)

## 4. Chia phiên nghe theo user

Sắp lượt nghe theo `uts` trong từng user. Lượt đầu mở phiên 1. Khoảng cách **đúng 30 phút** vẫn thuộc cùng phiên, **trên 30 phút** mở phiên mới. Một phiên có thể đi qua nửa đêm. Khóa phiên là cặp `(user, phien)`.

`khoang_phut` đo thời gian giữa scrobble đầu và cuối phiên. Một phiên một lượt có giá trị 0. Đây là độ dài quan sát được. Nếu crawl bỏ sót lượt ở giữa, một phiên thực có thể bị tách làm hai. Đọc kết quả cùng `so_ngay_co_nghe` ở bước 2.

In [ ]:
order_user = Window.partitionBy("user").orderBy("uts", "event_id")
running = order_user.rowsBetween(Window.unboundedPreceding, Window.currentRow)
session_plays = (
    plays.withColumn("gap_s", F.col("uts") - F.lag("uts").over(order_user))
    .withColumn("bat_dau", (F.col("gap_s").isNull() | (F.col("gap_s") > 1800)).cast("int"))
    .withColumn("phien", F.sum("bat_dau").over(running))
)
sessions = (
    session_plays.groupBy("user", "phien")
    .agg(F.min("ts_utc").alias("bat_dau_utc"), F.max("ts_utc").alias("ket_thuc_utc"),
         F.count("*").alias("luot_trong_phien"),
         F.round((F.max("uts") - F.min("uts")) / 60, 1).alias("khoang_phut"))
)
sessions_with_coverage = sessions.join(coverage.select("user", "so_ngay_co_nghe", "so_ngay_lich"), "user").orderBy(
    "user", "phien"
)
show_all(sessions_with_coverage)

## 5. Hoạt động tăng hay giảm qua các tuần UTC?

`tuan_utc` là ngày thứ Hai mở đầu tuần, gồm cả năm nên không gộp nhầm tuần giữa các năm. `share_pct` là tỷ trọng hoạt động của tuần trong toàn bộ mẫu, lấy trung bình đều theo user như bước 3.

Để đo thay đổi, chỉ so sánh những user **có lượt ở cả hai tuần lịch liên tiếp**. Với từng user, lấy tỷ trọng tuần này trừ tỷ trọng tuần trước rồi mới lấy trung bình. Dấu của `thay_doi_diem_pct` cho biết chiều thay đổi trong nhóm so sánh. Đọc kèm `so_user_so_sanh`. Giá trị trống nghĩa là chưa có cặp tuần đủ điều kiện. Hai tuần đầu/cuối được crawl một phần hoặc các tuần có lịch crawl khác nhau có thể làm sai lệch xu hướng.

In [ ]:
user_weeks = (
    plays.groupBy("user", "tuan_utc").count()
    .join(coverage.select("user", "so_luot"), "user")
    .withColumn("user_share", F.col("count") / F.col("so_luot"))
)
weekly = user_weeks.groupBy("tuan_utc").agg(
    F.sum("count").alias("so_luot"), F.count("*").alias("so_user_co_nghe"),
    F.round(100 * F.sum("user_share") / F.lit(n_users), 2).alias("share_pct"),
)
w_week = Window.partitionBy("user").orderBy("tuan_utc")
paired = (
    user_weeks.withColumn("tuan_truoc", F.lag("tuan_utc").over(w_week))
    .withColumn("share_truoc", F.lag("user_share").over(w_week))
    .where(F.datediff("tuan_utc", "tuan_truoc") == 7)
    .groupBy("tuan_utc").agg(F.count("*").alias("so_user_so_sanh"),
        F.round(100 * F.avg(F.col("user_share") - F.col("share_truoc")), 2).alias("thay_doi_diem_pct"))
)
trend = weekly.join(paired, "tuan_utc", "left").fillna(0, subset=["so_user_so_sanh"])
show_all(trend.orderBy("tuan_utc"))

## 6. Giờ UTC nào thường nghe tag gì?

Dùng `tags` và `tag_source` ngay trong Silver. Pipeline Silver đã ưu tiên tag bài hát, dự phòng tag nghệ sĩ và giữ tối đa 5 tag; notebook không chọn lại metadata từ Bronze. Do khác quy tắc chọn phiên bản/tag, kết quả có thể khác bản notebook đọc Bronze trước đây.

Dọn tag trắng/trùng trước khi tính tỷ lệ. Mỗi lượt đóng góp tổng trọng số 1: hai tag đóng góp 0,5 cho mỗi tag; lượt thiếu tag vào nhóm “(thiếu tag)”. Trong mỗi giờ, tính tỷ lệ của từng user rồi lấy trung bình đều giữa **các user có nghe trong giờ**. Tổng tỷ lệ các tag trong một giờ là 100%.

In [ ]:
cleaned_tags = F.array_distinct(F.filter(
    F.transform("tags", lambda tag: F.lower(F.trim(tag))),
    lambda tag: tag.isNotNull() & (tag != ""),
))
tagged = (
    plays.select("event_id", "user", "gio_utc", "tag_source", cleaned_tags.alias("tags"))
    .withColumn("tags", F.when(F.size("tags") > 0, F.col("tags")))
    .withColumn("tag_source", F.when(F.col("tags").isNull(), "missing").otherwise(F.col("tag_source")))
)
show_all(tagged.groupBy("tag_source").count().orderBy("tag_source"))

user_hour = Window.partitionBy("user", "gio_utc")
weighted = (
    tagged.withColumn("luot_user_gio", F.count("*").over(user_hour))
    .withColumn("labels", F.coalesce("tags", F.array(F.lit("(thiếu tag)"))))
    .withColumn("weight", 1.0 / (F.size("labels") * F.col("luot_user_gio")))
    .withColumn("tag", F.explode("labels"))
)
hour_user_counts = plays.groupBy("gio_utc").agg(F.countDistinct("user").alias("so_user_co_nghe"))
hour_tags = (
    weighted.groupBy("gio_utc", "tag").agg(F.sum("weight").alias("sum_share"))
    .join(hour_user_counts, "gio_utc")
    .withColumn("share_pct", 100 * F.col("sum_share") / F.col("so_user_co_nghe"))
    .drop("sum_share")
)
show_all(hour_tags.orderBy("gio_utc", F.desc("share_pct"), "tag"))

## 7. Xuất kết quả sang Gold trên MinIO

Chạy cell dưới để ghi kết quả. Mỗi lần chạy dùng thư mục mới, giữ các lần xuất trước. Cell kiểm tra số dòng và kiểu cột sau khi đọc lại Parquet, rồi ghi manifest thành công.


In [ ]:
tables = {
    "user_coverage": coverage,
    "hour_distribution": by_hour,
    "weekday_distribution": by_weekday,
    "month_distribution": by_month,
    "day_type_distribution": by_day_type,
    "day_comparison": day_comparison,
    "listening_sessions": sessions_with_coverage,
    "weekly_trend": trend,
    "hour_tags": hour_tags,
}



def write_gold_tables(tables, destination):
    """Ghi và đọc lại các bảng Parquet; trả về metadata sau khi kiểm tra."""
    exported_tables = {}
    for name, df in tables.items():
        path = f"{destination}/{name}"
        expected_rows = df.count()
        df.write.mode("errorifexists").option("compression", "snappy").parquet(path)
        restored = spark.read.parquet(path)
        actual_rows = restored.count()
        expected_types = [(field.name, field.dataType.simpleString()) for field in df.schema]
        actual_types = [(field.name, field.dataType.simpleString()) for field in restored.schema]
        if actual_rows != expected_rows or actual_types != expected_types:
            raise RuntimeError(f"Kiểm tra Gold thất bại ở {name}; lần xuất này chưa hoàn tất.")
        exported_tables[name] = {"path": path, "rows": actual_rows, "schema": restored.schema.jsonValue()}
        print(f"Đã lưu và đọc lại: {name} ({actual_rows} dòng)")
    return exported_tables


In [ ]:
import io
import json
from datetime import datetime, timezone
from uuid import uuid4

# Chỉ chạy cell này sau khi đã chạy đủ các cell phân tích phía trên.
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_" + uuid4().hex[:8]
run_prefix = "/".join(part for part in [gold_prefix, run_id] if part)
base_path = f"s3a://{gold_bucket}/{run_prefix}"
if not minio_client.bucket_exists(gold_bucket):
    minio_client.make_bucket(gold_bucket)

exported_tables = write_gold_tables(tables, base_path)

# Chỉ đánh dấu thành công sau khi đã kiểm tra đủ chín bảng.
manifest = {
    "status": "succeeded", "run_id": run_id,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": {"path": SILVER_PATH, "format": "delta", "version": silver_version, "rows": n_plays},
    "timezone": "UTC", "format": "parquet", "tables": exported_tables,
}
manifest_bytes = json.dumps(manifest, ensure_ascii=False, indent=2).encode("utf-8")
minio_client.put_object(gold_bucket, f"{run_prefix}/_manifest.json", io.BytesIO(manifest_bytes),
                       length=len(manifest_bytes), content_type="application/json")
print(f"Hoàn tất: {base_path}")

**Khi nối với bảng gợi ý:** cột `user` trong Gold là `user_id` của Silver. Giữ cùng khóa `artist_id`, `album_id`, `track_id` của Silver khi tạo thêm đặc trưng theo item. Nếu dùng các tỷ lệ để huấn luyện, lọc `uts` trước mốc test rồi tính lại các bảng trên tập train.

## 8. Đọc lại kết quả Gold

Trên MinIO Console, mở bucket Gold → prefix đã cấu hình → `run_id`. Mỗi bảng gồm các file `part-*.parquet`; đọc cả thư mục bảng bằng Spark. `_manifest.json` ghi phiên bản Silver, số dòng và schema của từng bảng. Nếu lần xuất bị gián đoạn thì chưa có manifest thành công; chạy lại cell xuất để tạo lần mới.

In [ ]:
saved_hour_distribution = spark.read.parquet(f"{base_path}/hour_distribution")
saved_hour_distribution.orderBy("gio_utc").show(24, truncate=False)